# Kaishoku ARG — Vercel Deploy Notebook

**Target**: `area_r_updated.zip`  
**Updated**: 2026-03-23 — タグシステム拡張（30タグ対応・新規テーブル8本追加）

実行順序: **STEP 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10**  
再デプロイ時: **STEP 0 → 2 → 3 → 4（→ 5 if schema変更あり）→ 7 → 10**


---
## STEP 0 — Settings (fill in before running anything)

In [ ]:
# ============================================================
# 全設定をここで入力する（他のセルは触らなくてOK）
# ============================================================

# --- GitHub ---
GITHUB_TOKEN      = "ghp_xxxxxxxxxxxxxxxxxxxx"  # repo スコープ必須
GITHUB_USER       = "your-username"             # GitHubユーザー名 or Org名
GITHUB_REPO       = "kaishoku"                  # リポジトリ名
GITHUB_BRANCH     = "main"                      # プッシュ先ブランチ
COMMIT_MESSAGE    = "deploy: tag system expansion v5 (30 tags, 8 new tables)"

# --- Vercel ---
VERCEL_TOKEN      = "xxxxxxxxxxxxxxxxxxxxxxxx"  # Vercel API Token
VERCEL_ORG_ID     = ""                          # チームID（個人アカウントは空文字でOK）
VERCEL_PROJECT_ID = ""                          # 空の場合は自動取得

# --- Turso / libSQL ---
TURSO_URL         = "libsql://your-db.turso.io" # TURSO_DATABASE_URL
TURSO_TOKEN       = "eyJhxxxxxxxxxxxxxxxx"       # TURSO_AUTH_TOKEN

# --- JWT / 内部シークレット ---
JWT_SECRET        = ""  # 空の場合はランダム生成
INTERNAL_SECRET   = ""  # 空の場合はランダム生成（32文字以上必須）
CRON_SECRET       = ""  # 空の場合はランダム生成

# --- 管理者パスワード ---
SEED_ADMIN_PASSWORD = ""  # 空の場合はランダム生成（8文字以上）

# --- ベースURL（デプロイ後のURL。空の場合は自動取得）---
DEPLOY_URL        = ""  # 例: https://kaishoku.vercel.app

# --- EmailJS（パスワードリセット OTP メール送信用）---
# https://www.emailjs.com/ でアカウント作成後に設定
# テンプレート変数: to_email / otp_code / agent_id / org_name
EMAILJS_SERVICE_ID  = ""  # 例: service_xxxxxxx（空の場合はメール送信をスキップ）
EMAILJS_TEMPLATE_ID = ""  # 例: template_xxxxxxx
EMAILJS_PUBLIC_KEY  = ""  # 例: xxxxxxxxxxxxxxxx

# --- zip ファイルのソース ---
ZIP_SOURCE        = "drive"   # "drive" or "upload"
ZIP_DRIVE_PATH    = "/content/drive/MyDrive/area_r_updated.zip"

# --- Eruda デバッグコンソール ---
ERUDA_ENABLED     = False   # True: Eruda を注入 / False: 無効（本番推奨）

# ============================================================
import secrets, string as _str

def _gen(n=32, extra=""):
    chars = _str.ascii_letters + _str.digits + extra
    return "".join(secrets.choice(chars) for _ in range(n))

if not JWT_SECRET or len(JWT_SECRET) < 32:
    JWT_SECRET = _gen(48)
    print(f"JWT_SECRET を自動生成しました: {JWT_SECRET}")

if not INTERNAL_SECRET or len(INTERNAL_SECRET) < 32:
    INTERNAL_SECRET = _gen(48)
    print(f"INTERNAL_SECRET を自動生成しました: {INTERNAL_SECRET}")

if not CRON_SECRET or len(CRON_SECRET) < 16:
    CRON_SECRET = _gen(32)
    print(f"CRON_SECRET を自動生成しました: {CRON_SECRET}")

if not SEED_ADMIN_PASSWORD or len(SEED_ADMIN_PASSWORD) < 8:
    SEED_ADMIN_PASSWORD = _gen(16, extra="!@#$%")
    print(f"SEED_ADMIN_PASSWORD を自動生成しました: {SEED_ADMIN_PASSWORD}")
    print("   上記パスワードを必ず控えてください！")

# ============================================================
# ユーティリティ実行フラグ（True にしたものだけ実行）
# ============================================================
RUN_ROLLBACK          = False
RUN_MIGRATE_ONLY      = False
RUN_GIT_LOG           = False
RUN_BACKUP            = False
RUN_DB_QUERY          = False
RUN_USER_MGMT         = False
RUN_CLEANUP           = False
RUN_ROTATE            = False
RUN_RATELIMIT_WATCH   = False
RUN_E2E               = False
RUN_DEPLOY_HISTORY    = False
RUN_PUSH_STAGING      = False
RUN_ALIAS             = False
RUN_MISSION_REVIEW    = False
RUN_FIRE_EVENT        = False
RUN_STORY_DIRECT      = False
RUN_TRANSFER_REVIEW   = False
RUN_ANALYTICS         = False
RUN_SEED_PUZZLES      = False
RUN_SEED_EVENT        = False
RUN_NPC_DM_CHECK      = False
RUN_CIPHER_STATS      = False
RUN_SEED_WORLD_DATA   = False  # 新テーブル（observation_points 等）初期データ投入
RUN_NOTIFY            = False
RUN_BULK_NOTIFY       = False
RUN_DM                = False

_util_flags = {
    "Rollback":               RUN_ROLLBACK,
    "Migrate only":           RUN_MIGRATE_ONLY,
    "Git log":                RUN_GIT_LOG,
    "DB backup":              RUN_BACKUP,
    "DB query":               RUN_DB_QUERY,
    "User mgmt":              RUN_USER_MGMT,
    "Log cleanup":            RUN_CLEANUP,
    "Rotate secrets":         RUN_ROTATE,
    "Rate limit watch":       RUN_RATELIMIT_WATCH,
    "E2E test":               RUN_E2E,
    "Deploy history":         RUN_DEPLOY_HISTORY,
    "Push staging":           RUN_PUSH_STAGING,
    "Set alias":              RUN_ALIAS,
    "Mission review":         RUN_MISSION_REVIEW,
    "Fire event":             RUN_FIRE_EVENT,
    "Story direct op":        RUN_STORY_DIRECT,
    "Transfer review":        RUN_TRANSFER_REVIEW,
    "Analytics":              RUN_ANALYTICS,
    "Send notify":            RUN_NOTIFY,
    "Bulk notify":            RUN_BULK_NOTIFY,
    "Admin DM":               RUN_DM,
}

eruda_status = "ON (STEP 2.5 inject)" if ERUDA_ENABLED else "OFF (production mode)"
print(f"\nEruda debug console: {eruda_status}")

enabled = [k for k, v in _util_flags.items() if v]
print(f"\nSettings complete — active utilities: {len(enabled)}")
for name in enabled:
    print(f"   • {name}")
if not enabled:
    print("   (none — deploy only)")


---
## STEP 1 — Environment Setup

In [ ]:
import subprocess, os, sys, json, zipfile, shutil, textwrap, time, re
from pathlib import Path
from datetime import datetime, timezone

def run(cmd, cwd=None, env=None, check=False):
    """Run a shell command and print output. Returns CompletedProcess."""
    print(f">>> {cmd[:120]}{'...' if len(cmd)>120 else ''}")
    e = {**os.environ, **(env or {})}
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd, env=e)
    if result.stdout.strip():
        print(result.stdout.strip()[:2000])
    if result.returncode != 0 and result.stderr.strip():
        print(f"STDERR: {result.stderr.strip()[:500]}")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (code {result.returncode}): {cmd}")
    return result

# Node.js 20
print("Setting up Node.js 20...")
run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1")
run("apt-get install -y nodejs > /dev/null 2>&1")
r = run("node --version && npm --version")
assert r.returncode == 0, "Node.js install failed"

# Git config
run('git config --global user.email "colab-deploy@kaishoku.local"')
run('git config --global user.name "Colab Deployer"')
run('git config --global init.defaultBranch main')
run('git config --global core.autocrlf false')

WORK_DIR = Path("/content/kaishoku-deploy")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("\nEnvironment setup complete")


---
## STEP 2 — Load & Extract zip

In [ ]:
ZIP_LOCAL = "/content/area_r_updated.zip"

if ZIP_SOURCE == "upload":
    from google.colab import files
    print("Please upload kaishoku.zip ...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.copy(fname, ZIP_LOCAL)
    print(f"Uploaded: {fname}")

elif ZIP_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    src = Path(ZIP_DRIVE_PATH)
    assert src.exists(), f"File not found: {ZIP_DRIVE_PATH}"
    shutil.copy(str(src), ZIP_LOCAL)
    print(f"Copied from Drive: {ZIP_DRIVE_PATH}")

# Extract zip
EXTRACT_DIR = WORK_DIR / "project"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_LOCAL, "r") as zf:
    zf.extractall(str(EXTRACT_DIR))

# Normalize: move nested project root up if needed
if not (EXTRACT_DIR / "src").exists():
    subdirs = [d for d in EXTRACT_DIR.iterdir() if d.is_dir()]
    for sub in subdirs:
        if (sub / "src").exists():
            for item in sub.iterdir():
                dest = EXTRACT_DIR / item.name
                if dest.exists():
                    if dest.is_dir(): shutil.rmtree(dest)
                    else: dest.unlink()
                shutil.move(str(item), str(EXTRACT_DIR))
            sub.rmdir()
            break

print("\nExtracted file structure:")
for p in sorted(EXTRACT_DIR.iterdir()):
    print(f"  {'[D]' if p.is_dir() else '[F]'} {p.name}")

# Validate required files for kaishoku.zip
assert (EXTRACT_DIR / "src").exists(),                              "Missing: src/"
assert (EXTRACT_DIR / "scripts" / "migrate.js").exists(),          "Missing: scripts/migrate.js"
assert (EXTRACT_DIR / "scripts" / "seed-from-area13.mjs").exists(),"Missing: scripts/seed-from-area13.mjs"
assert (EXTRACT_DIR / "scripts" / "schema.sql").exists(),          "Missing: scripts/schema.sql"
assert (EXTRACT_DIR / "scripts" / "area13").exists(),              "Missing: scripts/area13/"
assert (EXTRACT_DIR / "package.json").exists(),                    "Missing: package.json"

print("\nzip extraction complete")


---
## STEP 2.5 — Eruda Debug Console Injection

Set `ERUDA_ENABLED = True` in STEP 0 to inject [Eruda](https://github.com/liriliri/eruda) into `layout.tsx`.

> **WARNING**: Set back to `False` before production re-deploy.

In [ ]:
if not ERUDA_ENABLED:
    print("ERUDA_ENABLED=False — skipping (production mode)")
else:
    layout_path = EXTRACT_DIR / "src" / "app" / "layout.tsx"
    if not layout_path.exists():
        print("WARNING: layout.tsx not found. Run STEP 2 first.")
    else:
        original = layout_path.read_text(encoding="utf-8")
        if "eruda" in original.lower():
            print("Eruda already injected — skip.")
        else:
            ERUDA_COMPONENT = """
// --- Eruda debug console (injected by Colab deploy) ---
function ErudaLoader() {
  return (
    <script dangerouslySetInnerHTML={{ __html: `
      (function() {
        var s = document.createElement('script');
        s.src = 'https://cdn.jsdelivr.net/npm/eruda';
        s.onload = function() { eruda.init(); };
        document.head.appendChild(s);
      })();
    ` }} />
  );
}
"""
            patched = original
            m = re.search(r'^(export default )', patched, re.MULTILINE)
            if m:
                patched = patched[:m.start()] + ERUDA_COMPONENT + patched[m.start():]
            else:
                patched = patched + ERUDA_COMPONENT
            patched = re.sub(r'(<body[^>]*>)', r'\1\n        <ErudaLoader />', patched, count=1)
            layout_path.with_suffix(".tsx.bak").write_text(original, encoding="utf-8")
            layout_path.write_text(patched, encoding="utf-8")
            print(f"Eruda injected: {layout_path}")
            print("WARNING: set ERUDA_ENABLED=False before production re-deploy")


---
## STEP 3 — Push to GitHub

In [ ]:
import requests

REPO_URL      = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
REPO_URL_SAFE = f"https://***@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
GIT_DIR       = WORK_DIR / "repo"

headers_gh = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

# Check / create repository
r = requests.get(f"https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}", headers=headers_gh)
if r.status_code == 404:
    print(f"Repository {GITHUB_REPO} not found — creating...")
    rc = requests.post("https://api.github.com/user/repos", headers=headers_gh,
                       json={"name": GITHUB_REPO, "private": True,
                              "description": "Kaishoku ARG (Next.js + Turso)"})
    assert rc.status_code == 201, f"Failed to create repo: {rc.text[:300]}"
    print(f"Repository created: https://github.com/{GITHUB_USER}/{GITHUB_REPO}")
    time.sleep(3)
elif r.status_code == 200:
    print(f"Repository found: https://github.com/{GITHUB_USER}/{GITHUB_REPO}")
else:
    raise RuntimeError(f"GitHub API error: {r.status_code} {r.text[:200]}")

# git clone or init
if GIT_DIR.exists():
    shutil.rmtree(GIT_DIR)

print(f"\nCloning: {REPO_URL_SAFE}")
clone_result = run(f"git clone --depth=50 {REPO_URL} {GIT_DIR} 2>&1")
if clone_result.returncode != 0:
    print("No existing history -> git init")
    GIT_DIR.mkdir(parents=True)
    run(f"git -C {GIT_DIR} init")
    run(f"git -C {GIT_DIR} remote add origin {REPO_URL}")

branch_out = run(f"git -C {GIT_DIR} branch -a").stdout
if f"remotes/origin/{GITHUB_BRANCH}" in branch_out:
    run(f"git -C {GIT_DIR} checkout -B {GITHUB_BRANCH} origin/{GITHUB_BRANCH}")
else:
    run(f"git -C {GIT_DIR} checkout -B {GITHUB_BRANCH}")

# Copy files
print("\nCopying files...")
for item in EXTRACT_DIR.iterdir():
    if item.name in {".git", "node_modules", ".next", ".vercel"}:
        continue
    dest = GIT_DIR / item.name
    if item.is_dir():
        if dest.exists(): shutil.rmtree(dest)
        shutil.copytree(str(item), str(dest),
                        ignore=shutil.ignore_patterns("node_modules", ".next"))
    else:
        shutil.copy2(str(item), str(dest))

# .gitignore
(GIT_DIR / ".gitignore").write_text(textwrap.dedent("""
    node_modules/
    .next/
    .env.local
    .env*.local
    *.log
    .vercel/
    dist/
    tsconfig.tsbuildinfo
    _db_check.mjs
""").strip())

# Stage & commit
run(f"git -C {GIT_DIR} add -A")
status = run(f"git -C {GIT_DIR} status --short")
if not status.stdout.strip():
    print("No changes — already up to date")
    commit_hash = run(f"git -C {GIT_DIR} rev-parse HEAD").stdout.strip()
else:
    changed_count = len(status.stdout.strip().splitlines())
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    full_msg = f"{COMMIT_MESSAGE}\n\nDeployed via Colab at {ts}\nChanged files: {changed_count}"
    run(f'git -C {GIT_DIR} commit -m {json.dumps(full_msg)}')

commit_hash = run(f"git -C {GIT_DIR} rev-parse HEAD").stdout.strip()

# Push: force-with-lease -> rebase -> force
push_r = run(f"git -C {GIT_DIR} push -u origin {GITHUB_BRANCH} --force-with-lease 2>&1")
if push_r.returncode != 0:
    print("force-with-lease failed -> trying pull --rebase...")
    rebase_r = run(f"git -C {GIT_DIR} pull --rebase origin {GITHUB_BRANCH} 2>&1")
    if rebase_r.returncode == 0:
        push_r2 = run(f"git -C {GIT_DIR} push -u origin {GITHUB_BRANCH} 2>&1")
        if push_r2.returncode != 0:
            run(f"git -C {GIT_DIR} push -u origin {GITHUB_BRANCH} --force 2>&1", check=True)
    else:
        run(f"git -C {GIT_DIR} push -u origin {GITHUB_BRANCH} --force 2>&1", check=True)

print(f"\nGitHub push complete")
print(f"   repo  : https://github.com/{GITHUB_USER}/{GITHUB_REPO}")
print(f"   branch: {GITHUB_BRANCH}")
print(f"   commit: {commit_hash[:12]}")


---
## STEP 4 — Vercel Deploy

In [ ]:
# Vercel CLI
run("npm install -g vercel@latest --silent")

VERCEL_HEADERS = {
    "Authorization": f"Bearer {VERCEL_TOKEN}",
    "Content-Type": "application/json",
}

# Get project ID
project_id = VERCEL_PROJECT_ID
if not project_id:
    params = {"name": GITHUB_REPO}
    if VERCEL_ORG_ID:
        params["teamId"] = VERCEL_ORG_ID
    r = requests.get("https://api.vercel.com/v9/projects", headers=VERCEL_HEADERS, params=params)
    projects = r.json().get("projects", [])
    if projects:
        project_id = projects[0]["id"]
        print(f"Vercel project found: {projects[0]['name']} ({project_id})")
    else:
        print("Vercel project not found — will create on deploy.")

# Set environment variables
env_vars = {
    "TURSO_DATABASE_URL":  TURSO_URL,
    "TURSO_AUTH_TOKEN":    TURSO_TOKEN,
    "JWT_SECRET":          JWT_SECRET,
    "INTERNAL_SECRET":     INTERNAL_SECRET,
    "CRON_SECRET":         CRON_SECRET,
    "NEXT_PUBLIC_BASE_URL":         DEPLOY_URL or "",
    "NEXT_PUBLIC_EMAILJS_SERVICE_ID":  EMAILJS_SERVICE_ID if "EMAILJS_SERVICE_ID" in dir() else "",
    "NEXT_PUBLIC_EMAILJS_TEMPLATE_ID": EMAILJS_TEMPLATE_ID if "EMAILJS_TEMPLATE_ID" in dir() else "",
    "NEXT_PUBLIC_EMAILJS_PUBLIC_KEY":  EMAILJS_PUBLIC_KEY if "EMAILJS_PUBLIC_KEY" in dir() else "",
}

if project_id:
    print("\nSetting Vercel environment variables...")
    team_param = f"?teamId={VERCEL_ORG_ID}" if VERCEL_ORG_ID else ""

    existing_envs = requests.get(
        f"https://api.vercel.com/v10/projects/{project_id}/env{team_param}",
        headers=VERCEL_HEADERS
    ).json().get("envs", [])
    existing_map = {e["key"]: e["id"] for e in existing_envs}

    for key, value in env_vars.items():
        if not value:
            print(f"  SKIP {key} (not set)")
            continue
        payload = {"value": value, "target": ["production", "preview"], "type": "encrypted"}
        if key in existing_map:
            r = requests.patch(
                f"https://api.vercel.com/v10/projects/{project_id}/env/{existing_map[key]}{team_param}",
                headers=VERCEL_HEADERS, json=payload)
        else:
            r = requests.post(
                f"https://api.vercel.com/v10/projects/{project_id}/env{team_param}",
                headers=VERCEL_HEADERS, json={**payload, "key": key})
        status_icon = "OK" if r.status_code in (200, 201) else "FAIL"
        print(f"  [{status_icon}] {key} (HTTP {r.status_code})")
        if r.status_code not in (200, 201):
            print(f"     {r.text[:200]}")

    # Validate required
    print("\n  Verifying required env vars...")
    check_envs = requests.get(
        f"https://api.vercel.com/v10/projects/{project_id}/env{team_param}",
        headers=VERCEL_HEADERS
    ).json().get("envs", [])
    set_keys = {e["key"] for e in check_envs}
    required_keys = ["TURSO_DATABASE_URL", "TURSO_AUTH_TOKEN", "JWT_SECRET", "INTERNAL_SECRET"]
    for key in required_keys:
        print(f"  {'[OK]' if key in set_keys else '[MISSING]'} {key}")
    missing = [k for k in required_keys if k not in set_keys]
    if missing:
        raise RuntimeError(f"Missing env vars: {missing}")
else:
    print("\nProject ID unknown — skipping env var API setup.")
    print("Set these manually in Vercel dashboard:")
    for k, v in env_vars.items():
        masked = v[:20] + "..." if len(v) > 20 else v
        print(f"   {k} = {masked}")

# Fix vercel.json: ensure framework + Hobby-plan cron
vj_path = GIT_DIR / "vercel.json"
try:
    vj = json.loads(vj_path.read_text()) if vj_path.exists() else {}
except Exception:
    vj = {}
vj.setdefault("framework", "nextjs")
for cron in vj.get("crons", []):
    if cron.get("schedule", "").startswith("0 *"):
        cron["schedule"] = "0 3 * * *"
        print("  Fixed: vercel.json cron -> daily (Hobby plan)")
vj_path.write_text(json.dumps(vj, ensure_ascii=False, indent=2))

# Remove .vercel dir to avoid stale project links
vercel_link_dir = GIT_DIR / ".vercel"
if vercel_link_dir.exists():
    shutil.rmtree(vercel_link_dir)

# Deploy
os.chdir(str(GIT_DIR))
prod_flag = "--prod" if GITHUB_BRANCH == "main" else ""
print(f"\nDeploying... (this may take a few minutes)")
deploy_result = run(f"vercel deploy {prod_flag} --token={VERCEL_TOKEN} --yes 2>&1")

url_match = re.search(r"https://[\w\-\.]+\.vercel\.app", deploy_result.stdout)
if url_match:
    detected_url = url_match.group(0)
    if not DEPLOY_URL:
        DEPLOY_URL = detected_url
    print(f"\nVercel deploy complete")
    print(f"   URL: {detected_url}")

    # Update NEXT_PUBLIC_BASE_URL with final URL
    if project_id and DEPLOY_URL:
        team_param = f"?teamId={VERCEL_ORG_ID}" if VERCEL_ORG_ID else ""
        existing_envs2 = requests.get(
            f"https://api.vercel.com/v10/projects/{project_id}/env{team_param}",
            headers=VERCEL_HEADERS
        ).json().get("envs", [])
        bu_env = next((e for e in existing_envs2 if e["key"] == "NEXT_PUBLIC_BASE_URL"), None)
        if bu_env:
            requests.patch(
                f"https://api.vercel.com/v10/projects/{project_id}/env/{bu_env['id']}{team_param}",
                headers=VERCEL_HEADERS,
                json={"value": DEPLOY_URL, "target": ["production","preview"], "type": "encrypted"})
            print(f"   NEXT_PUBLIC_BASE_URL updated: {DEPLOY_URL}")
else:
    print("\nWARNING: Could not detect deploy URL automatically.")
    print("   Check https://vercel.com/dashboard")
    if not DEPLOY_URL:
        DEPLOY_URL = input("Enter deploy URL manually: ").strip()


---
## STEP 5 — Turso Migration (schema.sql)

In [ ]:
os.chdir(str(GIT_DIR))

print("Installing dependencies...")
run("npm install --save-dev @libsql/client bcryptjs 2>&1 | tail -5")

# .env.local 生成
env_content = "\n".join([
    f"TURSO_DATABASE_URL={TURSO_URL}",
    f"TURSO_AUTH_TOKEN={TURSO_TOKEN}",
    f"JWT_SECRET={JWT_SECRET}",
    f"INTERNAL_SECRET={INTERNAL_SECRET}",
    f"CRON_SECRET={CRON_SECRET}",
    f"SEED_ADMIN_PASSWORD={SEED_ADMIN_PASSWORD}",
    f"NEXT_PUBLIC_BASE_URL={DEPLOY_URL}",
    f"NEXT_PUBLIC_EMAILJS_SERVICE_ID={EMAILJS_SERVICE_ID}",
    f"NEXT_PUBLIC_EMAILJS_TEMPLATE_ID={EMAILJS_TEMPLATE_ID}",
    f"NEXT_PUBLIC_EMAILJS_PUBLIC_KEY={EMAILJS_PUBLIC_KEY}",
])
(GIT_DIR / ".env.local").write_text(env_content)
print(".env.local generated")

# migrate.js を実行（schema.sql を適用 — タグ拡張8テーブルを含む全テーブル）
print("\nRunning migration (schema.sql)...")
result = run("node --env-file=.env.local scripts/migrate.js 2>&1")
if result.returncode != 0:
    raise RuntimeError(f"Migration failed:\n{result.stdout}\n{result.stderr}")

# adminパスワードを migrate.js で設定（--admin-password オプション）
print("\nSetting admin password via migrate.js...")
result_pw = run(f"node --env-file=.env.local scripts/migrate.js --admin-password={SEED_ADMIN_PASSWORD} 2>&1")
if result_pw.returncode != 0:
    print(f"  WARNING: password set failed: {result_pw.stdout}")
else:
    print(f"  Admin password set: K-000-ADMIN")

print("\nMigration complete")
print("  - mission_participants table included")
print("  - achievements / user_achievements tables included")


---
## STEP 6 — Seed (area13 initial data)

| Data | File |
|------|------|
| Divisions (DIV-01~05) | `scripts/area13/divisions-data.json` |
| Entities (E-001~020) | `scripts/area13/entities-data.json` |
| Modules (M-001-alpha~M-020-upsilon) | `scripts/area13/modules-data.json` |
| Missions | `scripts/area13/mission-data.json` |
| Personnel | `scripts/area13/personnel-data.json` |
| Incidents (area-001~009) | `scripts/area13/map-incidents.json` |

> **WARNING**: Run this step only once on first deploy.

In [ ]:
# WARNING: Run this step only once on first deploy.
# Subsequent runs will INSERT OR REPLACE existing data.
RUN_SEED = True  # Set to False to skip on re-deploys

if RUN_SEED:
    print("Running seed (area13 data)...")
    result = run("node --env-file=.env.local scripts/seed-from-area13.mjs 2>&1")
    if result.returncode != 0:
        raise RuntimeError(f"Seed failed:\n{result.stdout}\n{result.stderr}")
    print("\nSeed complete (area13 data loaded)")
else:
    print("Seed skipped (RUN_SEED=False)")

# Reset admin password
print("\nSetting admin password...")
reset_script = GIT_DIR / "scripts" / "_reset_admin.mjs"
reset_script.write_text(
    "import { createClient } from '@libsql/client';\n"
    "import bcrypt from 'bcryptjs';\n"
    "const db = createClient({ url: process.env.TURSO_DATABASE_URL, authToken: process.env.TURSO_AUTH_TOKEN });\n"
    "async function run() {\n"
    "  const hash = await bcrypt.hash(process.env.SEED_ADMIN_PASSWORD, 12);\n"
    "  const r = await db.execute({\n"
    "    sql: \"UPDATE users SET password_hash=? WHERE agent_id='K-000-ADMIN' OR username='K-000-ADMIN'\",\n"
    "    args: [hash]\n"
    "  });\n"
    "  console.log('updated:', r.rowsAffected, 'rows');\n"
    "  if (r.rowsAffected === 0) console.log('WARNING: K-000-ADMIN not found');\n"
    "}\n"
    "run().catch(e => { console.error(e.message); process.exit(1); });\n"
)
r_reset = run("node --env-file=.env.local scripts/_reset_admin.mjs 2>&1")
reset_script.unlink(missing_ok=True)
if r_reset.returncode == 0:
    print(f"  Admin password set")
    print(f"  agentId : K-000-ADMIN")
    print(f"  password: {SEED_ADMIN_PASSWORD}")
    print("\n  *** Save this password now! ***")
else:
    print(f"  WARNING: Password reset failed: {r_reset.stdout}")


---
## STEP 7 — Health Check & API Tests

In [ ]:
import urllib.request, urllib.error

def api_check(label, url, method="GET", body=None, expected_status=200,
              headers=None, check_key=None, check_value=None, cookie=None):
    h = {"Content-Type": "application/json", "User-Agent": "Colab-Check/2.0", "X-Requested-With": "XMLHttpRequest"}
    if headers: h.update(headers)
    if cookie:  h["Cookie"] = cookie
    data = json.dumps(body).encode() if body else None
    req  = urllib.request.Request(url, data=data, headers=h, method=method)
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            status  = resp.status
            content = resp.read().decode(errors="replace")
            resp_headers = dict(resp.headers)
    except urllib.error.HTTPError as e:
        status  = e.code
        content = e.read().decode(errors="replace")
        resp_headers = dict(e.headers)
    except Exception as e:
        print(f"  FAIL {label}: connection error — {e}")
        return False, None

    try:
        rjson = json.loads(content)
    except Exception:
        rjson = {}

    ok = (status == expected_status)
    if check_key and ok:
        ok = (rjson.get(check_key) == check_value) if check_value is not None else (check_key in rjson)

    icon    = "[OK]" if ok else "[FAIL]"
    snippet = json.dumps(rjson, ensure_ascii=False)[:150] if rjson else content[:150]
    print(f"  {icon} [{status}] {label}")
    print(f"       -> {snippet}")
    return ok, resp_headers

base = DEPLOY_URL.rstrip("/")
if not base:
    base = input("Enter deploy URL (e.g. https://kaishoku.vercel.app): ").strip().rstrip("/")

print(f"\nChecking: {base}")
print("=" * 60)
results = {}

print("\n[Health check]")
ok, _ = api_check("/api/health", f"{base}/api/health", check_key="status", check_value="ok")
results["health"] = ok

print("\n[Auth guard — expect 401]")
for path in ["/api/users/me", "/api/skill-tree"]:
    ok, _ = api_check(path, f"{base}{path}", expected_status=401)
    results[f"auth_guard:{path}"] = ok

print("\n[Admin guard — expect 401]")
for path in ["/api/admin/users", "/api/admin/analytics"]:
    ok, _ = api_check(path, f"{base}{path}", expected_status=401)
    results[f"admin_guard:{path}"] = ok

print("\n[新規 API — タグシステム拡張 (expect 401 without auth)]")
new_apis = [
    "/api/audio",
    "/api/observation-points",
    "/api/observation-logs",
    "/api/dimension-cracks",
    "/api/case-reports",
    "/api/operation-records",
    "/api/containment-protocols",
    "/api/research-theories",
    "/api/divisions",
]
for path in new_apis:
    ok, _ = api_check(path, f"{base}{path}", expected_status=401)
    results[f"new_api_guard:{path}"] = ok

print("\n[Input validation]")
ok, _ = api_check("short password -> 400", f"{base}/api/auth/register",
    method="POST",
    body={"username":"K-TST-001","password":"abc",
          "divisionId":"DIV-02"},
    expected_status=400)
results["validation:short_password"] = ok

ok, _ = api_check("invalid agentId -> 401", f"{base}/api/auth/login",
    method="POST",
    body={"username":"K-000-INVALID","password":"wrongpassword"},
    expected_status=401)
results["validation:invalid_login"] = ok

print("\n[Rate limiting — expect 429 after repeated failures]")
rl_triggered = False
for i in range(12):
    expected = 429 if i >= 10 else 401
    ok, _ = api_check(f"login fail #{i+1}", f"{base}/api/auth/login",
        method="POST",
        body={"username":"K-RL-TEST","password":"wrongpw"},
        expected_status=expected)
    if i >= 5 and ok:
        rl_triggered = True
        break
results["rate_limit"] = rl_triggered

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
passed = sum(1 for v in results.values() if v)
total  = len(results)
for name, ok in results.items():
    print(f"  {'[OK]' if ok else '[FAIL]'} {name}")
print("=" * 60)
print(f"  Total: {passed}/{total} passed")
if passed == total:
    print("\nAll checks passed — deploy successful!")
else:
    print("\nSome checks failed. Review the logs.")


---
## STEP 8 — Admin Login Check

In [ ]:
ADMIN_AGENT_ID       = "K-000-ADMIN"
ADMIN_PASSWORD_CHECK = SEED_ADMIN_PASSWORD

print("Checking admin login...")
ok, resp_headers = api_check(
    "/api/auth/login (admin)",
    f"{base}/api/auth/login",
    method="POST",
    body={"username": ADMIN_AGENT_ID, "password": ADMIN_PASSWORD_CHECK},
    expected_status=200,
    check_key="ok",
    check_value=True,
)

if not ok:
    print("\nLogin failed — resetting password directly via DB...")
    reset_script2 = GIT_DIR / "scripts" / "_reset_admin2.mjs"
    reset_script2.write_text(
        "import { createClient } from '@libsql/client';\n"
        "import bcrypt from 'bcryptjs';\n"
        "const db = createClient({ url: process.env.TURSO_DATABASE_URL, authToken: process.env.TURSO_AUTH_TOKEN });\n"
        "async function run() {\n"
        "  const hash = await bcrypt.hash(process.env.SEED_ADMIN_PASSWORD, 12);\n"
        "  const r = await db.execute({\n"
        "    sql: \"UPDATE users SET password_hash=? WHERE agent_id='K-000-ADMIN' OR username='K-000-ADMIN'\",\n"
        "    args: [hash]\n"
        "  });\n"
        "  console.log('updated:', r.rowsAffected);\n"
        "}\n"
        "run().catch(e => { console.error(e.message); process.exit(1); });\n"
    )
    run("node --env-file=.env.local scripts/_reset_admin2.mjs 2>&1")
    reset_script2.unlink(missing_ok=True)
    print("Retrying login...")
    ok, resp_headers = api_check(
        "/api/auth/login (admin) [after reset]",
        f"{base}/api/auth/login",
        method="POST",
        body={"username": ADMIN_AGENT_ID, "password": ADMIN_PASSWORD_CHECK},
        expected_status=200, check_key="ok", check_value=True,
    )

session_cookie = ""
if ok:
    set_cookie = resp_headers.get("Set-Cookie", "") if resp_headers else ""
    if set_cookie:
        session_cookie = set_cookie.split(";")[0]

    ok2, _ = api_check(
        "/api/users/me (role check)",
        f"{base}/api/users/me",
        cookie=session_cookie,
        check_key="role",
        check_value="admin",
    )
    if ok2:
        print(f"\nAdmin login OK")
        print(f"   agentId : {ADMIN_AGENT_ID}")
        print(f"   password: {ADMIN_PASSWORD_CHECK}")
    else:
        print("\nWARNING: Login OK but role != super_admin")
else:
    print(f"\nAdmin login FAILED")
    print(f"   SEED_ADMIN_PASSWORD = {repr(SEED_ADMIN_PASSWORD)}")


---
## STEP 9 — Turso DB Direct Check

In [ ]:
check_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{
  url: '{TURSO_URL}',
  authToken: '{TURSO_TOKEN}',
}});

const SEP = String.fromCharCode(0x2500).repeat(50);

async function check() {{
  // Table list
  const tables = await db.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name");
  console.log('\\nTables (' + tables.rows.length + '):');
  tables.rows.forEach(r => console.log('  -', r[0]));

  // User stats
  const ustat = await db.execute(`
    SELECT
      COUNT(*) AS total,
      SUM(CASE WHEN role='super_admin' THEN 1 ELSE 0 END) AS super_admin,
      SUM(CASE WHEN role='admin' THEN 1 ELSE 0 END) AS admin,
      SUM(CASE WHEN role='player' THEN 1 ELSE 0 END) AS player
    FROM users
  `);
  const u = ustat.rows[0];
  console.log('\\nUsers: total=' + u[0] + ' (super_admin=' + u[1] + ', admin=' + u[2] + ', player=' + u[3] + ')');

  // Divisions
  const divs = await db.execute('SELECT id, name FROM divisions ORDER BY id');
  console.log('\\nDivisions:');
  divs.rows.forEach(r => console.log('  ' + r[0] + ': ' + r[1]));

  // Row counts
  console.log('\\n' + SEP);
  console.log('Row counts');
  console.log(SEP);
  const targets = [
    // 既存テーブル
    'users', 'divisions', 'posts', 'notifications',
    'chat_messages', 'progress_flags', 'xp_logs',
    'missions', 'db_entities', 'db_equipment',
    'mission_participants', 'achievements', 'user_achievements',
    'rule_engine_entries', 'npc_engine_rules',
    // タグ拡張テーブル（v4 追加）
    'audio_records',
    // タグ拡張テーブル（v5 追加）
    'observation_points', 'observation_logs',
    'dimension_cracks', 'agent_memos',
    'case_reports', 'operation_records',
    'containment_protocols', 'research_theories',
  ];
  for (const t of targets) {{
    try {{
      const r = await db.execute('SELECT COUNT(*) AS cnt FROM ' + t);
      console.log('  ' + t.padEnd(28) + r.rows[0][0]);
    }} catch (e) {{
      console.log('  ' + t.padEnd(28) + '(no table)');
    }}
  }}
}}

check().catch(e => {{ console.error('DB check failed:', e.message); process.exit(1); }});
"""

check_path = GIT_DIR / "_db_check.mjs"
check_path.write_text(check_code)

print("Checking DB directly...")
result = run(f"node {check_path} 2>&1")
check_path.unlink(missing_ok=True)

if result.returncode == 0:
    print("\nDB check complete")
else:
    print("\nDB check FAILED — verify TURSO_URL / TURSO_TOKEN")


---
## STEP 10 — Deploy Report

In [ ]:
now_str      = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
passed_count = sum(1 for v in results.values() if v)
total_count  = len(results)

report  = f"# Kaishoku Deploy Report\n\n"
report += f"**Date**: {now_str}\n\n"
report += "## Deploy Info\n"
report += "| Item | Value |\n|------|-------|\n"
report += f"| Repository | https://github.com/{GITHUB_USER}/{GITHUB_REPO} |\n"
report += f"| Branch | `{GITHUB_BRANCH}` |\n"
report += f"| Commit | `{commit_hash[:12]}` |\n"
report += f"| Message | {COMMIT_MESSAGE} |\n"
report += f"| Deploy URL | {DEPLOY_URL or '(check dashboard)'} |\n\n"

report += "## API Check Results\n"
report += f"**{passed_count}/{total_count} passed**\n\n"
report += "| Check | Result |\n|-------|--------|\n"
for name, ok in results.items():
    report += f"| {name} | {'PASS' if ok else 'FAIL'} |\n"

report += "\n## Seeded Data (area13)\n"
report += "| Data | Content |\n|------|----------|\n"
report += "| Divisions | DIV-01 Observation / DIV-02 Convergence / DIV-03 Record / DIV-04 Tech / DIV-05 Seal |\n"
report += "| Entities | E-001 ~ E-020 (20 items) |\n"
report += "| Modules | M-001-alpha ~ M-020-upsilon (20 items) |\n"
report += "| Missions | MISSION-2026-001~009, MISSION-2025-347 etc. |\n"
report += "| Incidents | area-001~009 (Oita Prefecture) |\n"

report += "\n## Environment Variables\n"
report += "| Key | Status |\n|-----|--------|\n"
for k in ["TURSO_DATABASE_URL", "TURSO_AUTH_TOKEN", "JWT_SECRET",
          "INTERNAL_SECRET", "CRON_SECRET", "NEXT_PUBLIC_BASE_URL"]:
    report += f"| {k} | set |\n"

print(report)

report_path = "/content/deploy_report.md"
Path(report_path).write_text(report)

try:
    from google.colab import files
    files.download(report_path)
    print("\nReport downloaded.")
except Exception:
    print(f"\nReport saved to: {report_path}")


---
## Utils — Rollback (emergency)

In [ ]:
if not RUN_ROLLBACK:
    print("RUN_ROLLBACK=False — skipped (enable in STEP 0)")
else:
    ROLLBACK_TARGET = "HEAD~1"  # change to specific hash if needed
    print(f"Rolling back to: {ROLLBACK_TARGET}")
    prev = run(f"git -C {GIT_DIR} rev-parse {ROLLBACK_TARGET}").stdout.strip()
    print(f"  Target commit: {prev[:12]}")
    run(f"git -C {GIT_DIR} revert HEAD --no-edit", check=True)
    run(f"git -C {GIT_DIR} push origin {GITHUB_BRANCH}", check=True)
    print("\nRollback commit created.")
    print("Vercel will automatically deploy the reverted code.")


---
## Utils — Migrate only (re-run schema)

In [ ]:
if not RUN_MIGRATE_ONLY:
    print("RUN_MIGRATE_ONLY=False — skipped (enable in STEP 0)")
else:
    print("Re-running migration only...")
    result = run(f"node --env-file=.env.local scripts/migrate.js --admin-password={SEED_ADMIN_PASSWORD} 2>&1", cwd=str(GIT_DIR))
    if result.returncode == 0:
        print("\nMigration re-run complete")
    else:
        print("\nMigration FAILED")


---
## Utils — Git log

In [ ]:
if not RUN_GIT_LOG:
    print("RUN_GIT_LOG=False — skipped (enable in STEP 0)")
else:
    print("=" * 60)
    print("Recent commit log (last 20)")
    print("=" * 60)
    run(f"git -C {GIT_DIR} log --oneline --graph --all -20")
    print("\nRemote branches:")
    run(f"git -C {GIT_DIR} branch -r")


---
## Utils — DB Backup (to Drive)

In [ ]:
BACKUP_TO_DRIVE  = True
BACKUP_DIR_DRIVE = "/content/drive/MyDrive/kaishoku_backups"

if not RUN_BACKUP:
    print("RUN_BACKUP=False — skipped (enable in STEP 0)")
else:
    backup_code = f"""
import {{ createClient }} from '@libsql/client';
import {{ writeFileSync, mkdirSync }} from 'fs';

const db = createClient({{
  url: '{TURSO_URL}',
  authToken: '{TURSO_TOKEN}',
}});

const TABLES = [
  'users','divisions','posts','notifications','bookmarks',
  'chat_messages','chat_read_markers','progress_flags','story_variables',
  'fired_events','xp_logs','mission_participants',
  'npc_engine_rules','rule_engine_entries',
  'access_logs','rate_limit_attempts',
  'missions','db_entities','db_equipment','db_facilities','db_personnel',
  'publish_queue','event_schedule','puzzle_entries','npc_dm_channels',
];

async function backup() {{
  const ts = new Date().toISOString().replace(/[:.]/g,'-').slice(0,19);
  const outDir = '/content/kaishoku_backup_' + ts;
  mkdirSync(outDir, {{ recursive: true }});
  const manifest = {{ timestamp: ts, tables: {{}} }};
  for (const table of TABLES) {{
    try {{
      const r = await db.execute('SELECT * FROM ' + table);
      const rows = r.rows.map(row => {{
        const obj = {{}};
        r.columns.forEach((col, i) => obj[col] = row[i]);
        return obj;
      }});
      writeFileSync(outDir + '/' + table + '.json', JSON.stringify(rows, null, 2));
      manifest.tables[table] = rows.length;
      console.log('  OK ' + table + ': ' + rows.length + ' rows');
    }} catch(e) {{
      console.log('  -- ' + table + ': skip (' + e.message.slice(0,50) + ')');
      manifest.tables[table] = null;
    }}
  }}
  writeFileSync(outDir + '/manifest.json', JSON.stringify(manifest, null, 2));
  console.log('\\nSaved to: ' + outDir);
  process.stdout.write('BACKUP_DIR=' + outDir + '\\n');
}}

backup().catch(e => {{ console.error('Backup failed:', e.message); process.exit(1); }});
"""
    bp_script = GIT_DIR / "_backup.mjs"
    bp_script.write_text(backup_code)
    print("Backing up...")
    result = run(f"node {bp_script} 2>&1")
    bp_script.unlink(missing_ok=True)

    bp_dir_match = re.search(r"BACKUP_DIR=(.+)", result.stdout)
    if bp_dir_match and result.returncode == 0:
        bp_dir = bp_dir_match.group(1).strip()
        bp_path = Path(bp_dir)
        import zipfile as _zf
        zip_name = f"/content/kaishoku_backup_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.zip"
        with _zf.ZipFile(zip_name, 'w', _zf.ZIP_DEFLATED) as zf:
            for f in bp_path.rglob('*'):
                zf.write(f, f.relative_to(bp_path.parent))
        size_kb = Path(zip_name).stat().st_size // 1024
        print(f"\nBackup complete: {zip_name} ({size_kb} KB)")
        if BACKUP_TO_DRIVE:
            try:
                from google.colab import drive as _drive
                _drive.mount("/content/drive", force_remount=False)
                import shutil as _sh
                Path(BACKUP_DIR_DRIVE).mkdir(parents=True, exist_ok=True)
                dest = f"{BACKUP_DIR_DRIVE}/{Path(zip_name).name}"
                _sh.copy2(zip_name, dest)
                print(f"Saved to Drive: {dest}")
            except Exception as e:
                print(f"Drive save skipped: {e}")
        try:
            from google.colab import files as _files
            _files.download(zip_name)
        except Exception:
            pass
    else:
        print("Backup FAILED")


---
## Utils — DB Management

### A. Arbitrary SQL / B. User management / C. Log cleanup

In [ ]:
if not RUN_DB_QUERY:
    print("RUN_DB_QUERY=False — skipped (enable in STEP 0)")
else:
    # Edit QUERY to run any SQL you need
    QUERY = """
      SELECT id, agent_id, role, status, clearance_level, xp_total, created_at
      FROM users

      ORDER BY created_at DESC
      LIMIT 20
    """
    query_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
async function run() {{
  const r = await db.execute(`{QUERY.strip()}`);
  console.log('columns:', r.columns.join(', '));
  console.log('rows:', r.rows.length);
  r.rows.forEach((row, i) => {{
    const obj = {{}};
    r.columns.forEach((col, j) => obj[col] = row[j]);
    console.log(i+1, JSON.stringify(obj));
  }});
}}
run().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    qs = GIT_DIR / "_query.mjs"
    qs.write_text(query_code)
    run(f"node {qs} 2>&1")
    qs.unlink(missing_ok=True)


In [ ]:
# ============================================================
# User management: ban / unban / suspend / activate / set_role
# ============================================================
TARGET_USERNAME = "K-000-XXXX"   # target agent_id
ACTION          = "ban"          # ban / unban / suspend / activate / set_role
NEW_ROLE        = "admin"        # used only for set_role

if not RUN_USER_MGMT:
    print("RUN_USER_MGMT=False — skipped (enable in STEP 0)")
else:
    STATUS_MAP = {"ban": "banned", "suspend": "suspended",
                  "unban": "active", "activate": "active"}
    if ACTION == "set_role":
        assert NEW_ROLE in ("player", "admin", "super_admin"), "Invalid role"
        sql = f"UPDATE users SET role = '{NEW_ROLE}' WHERE agent_id = '{TARGET_USERNAME}' "
    elif ACTION in STATUS_MAP:
        sql = f"UPDATE users SET status = '{STATUS_MAP[ACTION]}' WHERE agent_id = '{TARGET_USERNAME}' "
    else:
        raise ValueError(f"Invalid action: {ACTION}")

    mgmt_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
async function run() {{
  const r = await db.execute(`{sql}`);
  console.log('rowsAffected:', r.rowsAffected);
  const check = await db.execute(`SELECT agent_id, role, status FROM users WHERE agent_id = '{TARGET_USERNAME}'`);
  if (check.rows.length) {{
    const u = check.rows[0];
    console.log('current state:', check.columns.map((c,i) => c + '=' + u[i]).join(', '));
  }} else {{
    console.log('WARNING: user not found');
  }}
}}
run().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    ms = GIT_DIR / "_mgmt.mjs"
    ms.write_text(mgmt_code)
    result = run(f"node {ms} 2>&1")
    ms.unlink(missing_ok=True)
    if result.returncode == 0:
        print(f"\n{ACTION} complete: {TARGET_USERNAME}")
    else:
        print("\nFAILED")


In [ ]:
CLEANUP_DAYS = 30  # delete logs older than N days

if not RUN_CLEANUP:
    print(f"RUN_CLEANUP=False — skipped (target: older than {CLEANUP_DAYS} days)")
else:
    cleanup_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
const DAYS = {CLEANUP_DAYS};
const targets = [
  {{ table: 'rate_limit_attempts', col: 'attempted_at' }},
  {{ table: 'access_logs',          col: 'created_at' }},
  {{ table: 'xp_logs',              col: 'created_at' }},
];
async function cleanup() {{
  for (const t of targets) {{
    try {{
      const r = await db.execute(
        `DELETE FROM ${{t.table}} WHERE ${{t.col}} < datetime('now', '-${{DAYS}} days')`
      );
      console.log('  OK ' + t.table + ': ' + r.rowsAffected + ' deleted');
    }} catch(e) {{
      console.log('  -- ' + t.table + ': skip (' + e.message.slice(0,50) + ')');
    }}
  }}
}}
cleanup().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    cs = GIT_DIR / "_cleanup.mjs"
    cs.write_text(cleanup_code)
    result = run(f"node {cs} 2>&1")
    cs.unlink(missing_ok=True)
    if result.returncode == 0:
        print(f"\nCleanup complete (logs older than {CLEANUP_DAYS} days deleted)")
    else:
        print("\nCleanup FAILED")


---
## Utils — Security: Rotate Secrets & Rate Limit Watch

In [ ]:
import secrets, string as _str_sec

def gen_secret(n=48):
    return ''.join(secrets.choice(_str_sec.ascii_letters + _str_sec.digits) for _ in range(n))

NEW_JWT_SECRET  = gen_secret(48)
NEW_CRON_SECRET = gen_secret(32)

if not RUN_ROTATE:
    print("RUN_ROTATE=False — skipped")
    print("\nGenerated values (copy if needed):")
    print(f"  JWT_SECRET  = {NEW_JWT_SECRET}")
    print(f"  CRON_SECRET = {NEW_CRON_SECRET}")
else:
    print("WARNING: Rotating JWT_SECRET will force-logout all users.")
    confirm = input("Continue? (yes/no): ").strip().lower()
    if confirm != "yes":
        print("Cancelled.")
    else:
        VH = {"Authorization": f"Bearer {VERCEL_TOKEN}", "Content-Type": "application/json"}
        team_param = f"?teamId={VERCEL_ORG_ID}" if VERCEL_ORG_ID else ""
        r = requests.get(
            f"https://api.vercel.com/v10/projects/{project_id}/env{team_param}",
            headers=VH)
        envs = r.json().get("envs", [])
        updates = {"JWT_SECRET": NEW_JWT_SECRET, "CRON_SECRET": NEW_CRON_SECRET}
        for key, value in updates.items():
            match = next((e for e in envs if e["key"] == key), None)
            if match:
                r2 = requests.patch(
                    f"https://api.vercel.com/v10/projects/{project_id}/env/{match['id']}{team_param}",
                    headers=VH,
                    json={"value": value, "target": ["production","preview"], "type": "encrypted"})
                print(f"  {'[OK]' if r2.status_code == 200 else '[FAIL]'} {key} (HTTP {r2.status_code})")
            else:
                print(f"  [SKIP] {key} not found — run STEP 4 first")
        print(f"\nRotation complete")
        print(f"  New JWT_SECRET  = {NEW_JWT_SECRET}")
        print(f"  New CRON_SECRET = {NEW_CRON_SECRET}")
        print("\nWARNING: Old/new tokens coexist until Vercel re-deploy completes.")


In [ ]:
if not RUN_RATELIMIT_WATCH:
    print("RUN_RATELIMIT_WATCH=False — skipped (enable in STEP 0)")
else:
    RATELIMIT_HOURS = 24
    watch_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
const HOURS = {RATELIMIT_HOURS};
const SEP = '-'.repeat(55);

async function watch() {{
  // Rate limit hits
  try {{
    const rl = await db.execute(`
      SELECT key_value, key_type, COUNT(*) AS attempts,
             MAX(attempted_at) AS last_at
      FROM rate_limit_attempts
      WHERE attempted_at > datetime('now', '-' || ${{HOURS}} || ' hours')
      GROUP BY key_value, key_type
      ORDER BY attempts DESC LIMIT 20
    `);
    console.log('\\nRate limit hits TOP 20 (last ' + HOURS + 'h):');
    console.log(SEP);
    rl.rows.forEach(r => {{
      const obj = {{}}; rl.columns.forEach((c,i) => obj[c]=r[i]);
      console.log('  ' + obj.key_type.padEnd(10) + ' | ' + String(obj.key_value).padEnd(20) + ' | ' + obj.attempts + ' attempts  last=' + obj.last_at);
    }});
    if (!rl.rows.length) console.log('  (none)');
  }} catch(e) {{ console.log('  rate_limit_attempts: ' + e.message.slice(0,60)); }}

  // High anomaly score users
  try {{
    const an = await db.execute(`
      SELECT agent_id, anomaly_score, observer_load, status
      FROM users
      WHERE status = 'active' AND anomaly_score > 10
      ORDER BY anomaly_score DESC LIMIT 10
    `);
    console.log('\\nHigh anomaly score users (> 10):');
    console.log(SEP);
    an.rows.forEach(r => {{
      const obj = {{}}; an.columns.forEach((c,i) => obj[c]=r[i]);
      console.log('  ' + String(obj.agent_id).padEnd(20) + ' score=' + Number(obj.anomaly_score).toFixed(1) + ' load=' + Number(obj.observer_load).toFixed(1) + ' status=' + obj.status);
    }});
    if (!an.rows.length) console.log('  (none)');
  }} catch(e) {{ console.log('  anomaly check: ' + e.message.slice(0,60)); }}
}}

watch().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    ws = GIT_DIR / "_watch.mjs"
    ws.write_text(watch_code)
    run(f"node {ws} 2>&1")
    ws.unlink(missing_ok=True)


---
## Utils — E2E Flow Test

In [ ]:
if not RUN_E2E:
    print("RUN_E2E=False — skipped (enable in STEP 0)")
else:
    import http.cookiejar

    E2E_AGENT_ID  = "K-E2E-TST"
    E2E_PASSWORD  = "E2eTestPass123!"
    E2E_DIVISION  = "DIV-02"

    class CookieSession:
        def __init__(self):
            self.jar = http.cookiejar.CookieJar()
            self.opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(self.jar))

        def request(self, url, method="GET", body=None, headers=None):
            h = {"Content-Type": "application/json", "User-Agent": "E2E-Test/1.0"}
            if headers: h.update(headers)
            data = json.dumps(body).encode() if body else None
            req = urllib.request.Request(url, data=data, headers=h, method=method)
            try:
                resp = self.opener.open(req, timeout=20)
                return resp.status, json.loads(resp.read().decode())
            except urllib.error.HTTPError as e:
                try: return e.code, json.loads(e.read().decode())
                except: return e.code, {}
            except Exception as e:
                return 0, {"error": str(e)}

    session = CookieSession()
    e2e_results = {}
    e2e_user_id = None

    def e2e_check(label, status, body, expect_status=200, check_key=None, check_val=None):
        ok = (status == expect_status)
        if check_key and ok:
            ok = (body.get(check_key) == check_val) if check_val is not None else (check_key in body)
        icon = "[OK]" if ok else "[FAIL]"
        snippet = json.dumps(body, ensure_ascii=False)[:120]
        print(f"  {icon} [{status}] {label}")
        print(f"       -> {snippet}")
        e2e_results[label] = ok
        return ok

    print(f"\nE2E test: {base}")
    print("=" * 60)

    print("\n[1] Register")
    status, body = session.request(f"{base}/api/auth/register", method="POST", body={
        "username": E2E_AGENT_ID,
        "password": E2E_PASSWORD, "divisionId": E2E_DIVISION,
    })
    e2e_check("Register -> 201", status, body, expect_status=201, check_key="ok", check_val=True)

    print("\n[2] Login")
    status, body = session.request(f"{base}/api/auth/login", method="POST",
                                   body={"username": E2E_AGENT_ID, "password": E2E_PASSWORD})
    login_ok = e2e_check("Login -> 200", status, body, expect_status=200, check_key="ok", check_val=True)

    if login_ok:
        print("\n[3] Get user info")
        status, body = session.request(f"{base}/api/users/me")
        me_ok = e2e_check("/api/users/me -> 200", status, body, check_key="agentId", check_val=E2E_AGENT_ID)
        if me_ok:
            e2e_user_id = body.get("id")

        print("\n[4] Set flag")
        status, body = session.request(f"{base}/api/users/me/flags", method="POST",
                                       body={"key": "e2e_test_flag", "value": True})
        e2e_check("Set flag -> 200", status, body, check_key="ok", check_val=True)

        print("\n[5] Logout")
        status, body = session.request(f"{base}/api/auth/logout", method="POST")
        e2e_results["Logout -> 200"] = (status == 200)
        print(f"  {'[OK]' if status==200 else '[FAIL]'} [{status}] Logout")

        print("\n[6] Verify logout (expect 401)")
        status, body = session.request(f"{base}/api/users/me")
        e2e_check("After logout /me -> 401", status, body, expect_status=401)

    # Clean up test user
    if e2e_user_id:
        print(f"\nDeleting test user: {E2E_AGENT_ID}")
        cleanup_e2e = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
async function run() {{
  await db.execute({{ sql: "DELETE FROM users WHERE id=?", args: ['{e2e_user_id}'] }});
  console.log('deleted');
}}
run().catch(e => console.error(e.message));
"""
        ce = GIT_DIR / "_cleanup_e2e.mjs"
        ce.write_text(cleanup_e2e)
        run(f"node {ce} 2>&1")
        ce.unlink(missing_ok=True)

    print("\n" + "=" * 60)
    e2e_passed = sum(1 for v in e2e_results.values() if v)
    e2e_total  = len(e2e_results)
    for name, ok in e2e_results.items():
        print(f"  {'[OK]' if ok else '[FAIL]'} {name}")
    print("=" * 60)
    print(f"  Total: {e2e_passed}/{e2e_total} passed")
    if e2e_passed == e2e_total:
        print("\nAll E2E tests passed!")
    else:
        print("\nSome E2E tests failed.")


---
## Utils — Vercel Deploy History & Staging

In [ ]:
if not RUN_DEPLOY_HISTORY:
    print("RUN_DEPLOY_HISTORY=False — skipped (enable in STEP 0)")
else:
    VH = {"Authorization": f"Bearer {VERCEL_TOKEN}", "Content-Type": "application/json"}
    team_q = f"&teamId={VERCEL_ORG_ID}" if VERCEL_ORG_ID else ""

    if project_id:
        r = requests.get(
            f"https://api.vercel.com/v6/deployments?projectId={project_id}&limit=10{team_q}",
            headers=VH)
        deployments = r.json().get("deployments", [])
        print("Recent Vercel deployments")
        print("=" * 70)
        for d in deployments:
            state_icon = {"READY":"[OK]","ERROR":"[FAIL]","BUILDING":"[...]","CANCELED":"[--]"}.get(d.get("state",""),"[?]")
            created = d.get("createdAt", 0)
            created_str = datetime.fromtimestamp(created/1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC") if created else "?"
            print(f"  {state_icon} {d.get('uid','')[:20]} | {d.get('state','').ljust(10)} | {created_str}")
            print(f"     URL: https://{d.get('url','')}")
            print(f"     Msg: {d.get('meta',{}).get('githubCommitMessage','')[:60]}")
            print()
    else:
        print("project_id not set — run STEP 4 first.")

# Staging push
STAGING_BRANCH = "staging"

if RUN_PUSH_STAGING:
    run(f"git -C {GIT_DIR} checkout -B {STAGING_BRANCH}")
    run(f"git -C {GIT_DIR} push -u origin {STAGING_BRANCH} --force-with-lease 2>&1")
    print(f"\nPushed to {STAGING_BRANCH} branch — Vercel preview deploy triggered.")
    run(f"git -C {GIT_DIR} checkout {GITHUB_BRANCH}")
else:
    print("RUN_PUSH_STAGING=False — skipped")

# Set alias
ALIAS_DEPLOYMENT_ID = ""   # e.g. "dpl_xxxxxxxxx"
ALIAS_DOMAIN        = ""   # e.g. "stable.kaishoku.vercel.app"

if RUN_ALIAS and ALIAS_DEPLOYMENT_ID and ALIAS_DOMAIN:
    VH2 = {"Authorization": f"Bearer {VERCEL_TOKEN}", "Content-Type": "application/json"}
    team_p = f"?teamId={VERCEL_ORG_ID}" if VERCEL_ORG_ID else ""
    r = requests.post(
        f"https://api.vercel.com/v2/deployments/{ALIAS_DEPLOYMENT_ID}/aliases{team_p}",
        headers=VH2, json={"alias": ALIAS_DOMAIN})
    if r.status_code == 200:
        print(f"Alias set: {ALIAS_DOMAIN} -> {ALIAS_DEPLOYMENT_ID}")
    else:
        print(f"Alias FAILED: {r.status_code} {r.text[:200]}")
else:
    print("RUN_ALIAS=False or ID/domain not set — skipped")


---
## Utils — Analytics Dashboard

In [ ]:
if not RUN_ANALYTICS:
    print("RUN_ANALYTICS=False — skipped (enable in STEP 0)")
else:
    analytics_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
const BAR = (n, max, width=30) => String.fromCharCode(0x2588).repeat(Math.round(n/Math.max(max,1)*width)).padEnd(width);
const SEP = '-'.repeat(60);

async function analyze() {{
  // DAU / WAU / MAU
  const dau = await db.execute(`
    SELECT
      SUM(CASE WHEN last_login_at > datetime('now','-1 day')  THEN 1 ELSE 0 END) AS dau,
      SUM(CASE WHEN last_login_at > datetime('now','-7 days') THEN 1 ELSE 0 END) AS wau,
      SUM(CASE WHEN last_login_at > datetime('now','-30 days')THEN 1 ELSE 0 END) AS mau,
      COUNT(*) AS total
    FROM users
  `);
  const s = dau.rows[0];
  console.log('\\nActive users');
  console.log(SEP);
  console.log('  DAU  (24h): ' + s[0] + ' / ' + s[3]);
  console.log('  WAU  (7d) : ' + s[1] + ' / ' + s[3]);
  console.log('  MAU  (30d): ' + s[2] + ' / ' + s[3]);

  // Clearance level distribution
  const levels = await db.execute(`
    SELECT clearance_level AS lv, COUNT(*) AS cnt
    FROM users GROUP BY lv ORDER BY lv ASC
  `);
  console.log('\\nClearance level distribution');
  console.log(SEP);
  const maxL = Math.max(...levels.rows.map(r=>Number(r[1])), 1);
  levels.rows.forEach(r => {{
    console.log('  Lv' + String(r[0]).padStart(2) + ' ' + BAR(r[1], maxL, 25) + ' ' + r[1]);
  }});

  // XP ranking TOP 15
  const xp = await db.execute(`
    SELECT agent_id, xp_total, clearance_level
    FROM users WHERE role != 'npc' ORDER BY xp_total DESC LIMIT 15
  `);
  console.log('\\nXP Ranking TOP 15');
  console.log(SEP);
  const maxX = Math.max(...xp.rows.map(r=>Number(r[1])), 1);
  xp.rows.forEach((r, i) => {{
    const rank = String(i+1).padStart(2);
    const name = String(r[0]).padEnd(20);
    console.log('  ' + rank + '. ' + name + ' Lv' + String(r[2]).padStart(2) + ' XP=' + String(r[1]).padStart(7) + '  ' + BAR(r[1], maxX, 15));
  }});

  // Division user count
  const divs = await db.execute(`
    SELECT d.name, COUNT(u.id) AS cnt FROM divisions d
    LEFT JOIN users u ON u.division_id = d.id AND u.role != 'npc'
    GROUP BY d.id ORDER BY cnt DESC
  `);
  console.log('\\nUsers per division');
  console.log(SEP);
  const maxDiv = Math.max(...divs.rows.map(r=>Number(r[1])), 1);
  divs.rows.forEach(r => {{
    console.log('  ' + String(r[0]).padEnd(14) + ' ' + BAR(r[1], maxDiv, 20) + ' ' + r[1]);
  }});

  // Anomaly score distribution
  const anom = await db.execute(`
    SELECT
      SUM(CASE WHEN anomaly_score < 20 THEN 1 ELSE 0 END)           AS safe,
      SUM(CASE WHEN anomaly_score BETWEEN 20 AND 49 THEN 1 ELSE 0 END) AS caution,
      SUM(CASE WHEN anomaly_score BETWEEN 50 AND 79 THEN 1 ELSE 0 END) AS warning,
      SUM(CASE WHEN anomaly_score >= 80 THEN 1 ELSE 0 END)           AS danger
    FROM users
  `);
  const a = anom.rows[0];
  console.log('\\nAnomaly score distribution');
  console.log(SEP);
  const labels = [['0-19  (safe)   ', a[0]], ['20-49 (caution)', a[1]], ['50-79 (warning)', a[2]], ['80+   (danger) ', a[3]]];
  const maxA = Math.max(...labels.map(l=>Number(l[1])), 1);
  labels.forEach(([label, cnt]) => {{
    console.log('  ' + label + ' ' + BAR(cnt, maxA, 20) + ' ' + cnt);
  }});

  // Recent XP grants (last 24h)
  const recentXp = await db.execute(`
    SELECT u.agent_id, xl.activity, xl.xp_gained, xl.created_at
    FROM xp_logs xl JOIN users u ON u.id = xl.user_id
    WHERE xl.created_at > datetime('now','-1 day')
    ORDER BY xl.created_at DESC LIMIT 10
  `);
  console.log('\\nRecent XP grants (24h)');
  console.log(SEP);
  recentXp.rows.forEach(r => {{
    console.log('  ' + String(r[0]).padEnd(20) + ' +' + String(r[2]).padStart(5) + ' XP  ' + r[1] + '  ' + String(r[3]).slice(5,16));
  }});
  if (!recentXp.rows.length) console.log('  (none)');
}}

analyze().catch(e=>{{console.error(e.message);process.exit(1);}});
"""
    as_ = GIT_DIR / "_analytics.mjs"
    as_.write_text(analytics_code)
    print("Fetching analytics data...")
    run(f"node {as_} 2>&1")
    as_.unlink(missing_ok=True)


---
## Utils — Notifications & ARG Game Management

In [ ]:
# ============================================================
# Send notification (all / division / level / user)
# target: "all" / "division:DIV-02" / "level:3" / "user:<UUID>"
# type:   info / warning / error / xp / levelup / admin_dm / critical
# ============================================================
NOTIFY_CONFIG = {
    "title":     "[NOTICE] System maintenance",
    "body":      "Maintenance scheduled: XXXX-XX-XX 02:00-04:00",
    "target":    "all",
    "type":      "info",
    "expiresAt": "",
}

if not RUN_NOTIFY:
    print("RUN_NOTIFY=False — skipped")
    print("\nPreview:")
    for k, v in NOTIFY_CONFIG.items():
        print(f"  {k:12} = {v}")
else:
    body = {k: v for k, v in NOTIFY_CONFIG.items() if v and k != "expiresAt"}
    if NOTIFY_CONFIG["expiresAt"]:
        body["expiresAt"] = NOTIFY_CONFIG["expiresAt"]
    ok, _ = api_check(
        f"Notify -> {NOTIFY_CONFIG['target']}",
        f"{base}/api/admin/notifications",
        method="POST", body=body,
        expected_status=200,
        cookie=session_cookie if "session_cookie" in dir() else "",
        check_key="ok", check_value=True,
    )
    if ok:
        print(f"\nNotification sent to: {NOTIFY_CONFIG['target']}")

# ============================================================
# Bulk notify (per division)
# ============================================================
BULK_NOTIFICATIONS = [
    {"title": "DIV-01 Notice", "body": "Message for observation division.", "target": "division:DIV-01", "type": "info"},
    {"title": "DIV-02 Notice", "body": "Message for convergence division.", "target": "division:DIV-02", "type": "info"},
    {"title": "DIV-03 Notice", "body": "Message for record division.",      "target": "division:DIV-03", "type": "info"},
    {"title": "DIV-04 Notice", "body": "Message for tech division.",        "target": "division:DIV-04", "type": "info"},
    {"title": "DIV-05 Notice", "body": "Message for seal division.",        "target": "division:DIV-05", "type": "info"},
]

if not RUN_BULK_NOTIFY:
    print("RUN_BULK_NOTIFY=False — skipped")
else:
    print(f"Bulk notify ({len(BULK_NOTIFICATIONS)} messages)")
    print("=" * 60)
    success = 0
    for notif in BULK_NOTIFICATIONS:
        ok, _ = api_check(
            f"-> {notif['target']}",
            f"{base}/api/admin/notifications",
            method="POST", body=notif,
            expected_status=200,
            cookie=session_cookie if "session_cookie" in dir() else "",
            check_key="ok", check_value=True,
        )
        if ok: success += 1
        time.sleep(0.3)
    print(f"\n{success}/{len(BULK_NOTIFICATIONS)} sent")


In [ ]:
# ============================================================
# ARG Game Management
# A. Mission application review
# B. Fire event manually
# C. Direct story flag/variable manipulation
# D. Division transfer review
# Run STEP 8 first (requires session_cookie)
# ============================================================

# --- A. Mission review ---
PARTICIPANT_ID   = ""            # fill from API GET result
MISSION_ACTION   = "approved"    # "approved" or "rejected"

if RUN_MISSION_REVIEW and PARTICIPANT_ID:
    ok, _ = api_check(
        f"Mission {MISSION_ACTION}: {PARTICIPANT_ID}",
        f"{base}/api/admin/mission-participants",
        method="PATCH",
        body={"participantId": PARTICIPANT_ID, "status": MISSION_ACTION},
        expected_status=200,
        cookie=session_cookie if "session_cookie" in dir() else "",
        check_key="ok", check_value=True,
    )
    if ok:
        print(f"\nMission {PARTICIPANT_ID} -> {MISSION_ACTION}")
else:
    print("Mission review skipped (RUN_MISSION_REVIEW=False or PARTICIPANT_ID not set)")

# --- B. Fire event ---
FIRE_EVENT_CONFIG = {
    "userId":   "",
    "eventId":  "chapter1_end",
    "xp":       500,
    "flag":     "chapter1_cleared",
    "flagValue": True,
    "notification": {
        "title": "Chapter 1 cleared",
        "body":  "Chapter 1 complete. Chapter 2 unlocked."
    }
}

if RUN_FIRE_EVENT and FIRE_EVENT_CONFIG["userId"]:
    ok, _ = api_check(
        f"Fire event: {FIRE_EVENT_CONFIG['eventId']}",
        f"{base}/api/admin/fire-event",
        method="POST", body=FIRE_EVENT_CONFIG,
        expected_status=200,
        cookie=session_cookie if "session_cookie" in dir() else "",
        check_key="ok", check_value=True,
    )
    if ok:
        print(f"\nEvent fired: {FIRE_EVENT_CONFIG['eventId']} -> user {FIRE_EVENT_CONFIG['userId'][:8]}...")
else:
    print("Event fire skipped (RUN_FIRE_EVENT=False or userId not set)")

# --- C. Story flags direct DB operation ---
STORY_ACTION   = "get"   # get / set_flag / set_var / del_flag / del_var / reset_all
STORY_USER_ID  = ""
STORY_KEY      = ""
STORY_VALUE    = "true"

if RUN_STORY_DIRECT and STORY_USER_ID:
    ops = {
        "get":       f"SELECT flag_key, flag_value, set_at FROM progress_flags WHERE user_id='{STORY_USER_ID}' ORDER BY set_at DESC",
        "set_flag":  f"INSERT OR REPLACE INTO progress_flags (user_id,flag_key,flag_value,set_at) VALUES ('{STORY_USER_ID}','{STORY_KEY}','{STORY_VALUE}',datetime('now'))",
        "set_var":   f"INSERT OR REPLACE INTO story_variables (user_id,var_key,var_value) VALUES ('{STORY_USER_ID}','{STORY_KEY}',{STORY_VALUE})",
        "del_flag":  f"DELETE FROM progress_flags WHERE user_id='{STORY_USER_ID}' AND flag_key='{STORY_KEY}'",
        "del_var":   f"DELETE FROM story_variables WHERE user_id='{STORY_USER_ID}' AND var_key='{STORY_KEY}'",
        "reset_all": f"DELETE FROM progress_flags WHERE user_id='{STORY_USER_ID}'; DELETE FROM story_variables WHERE user_id='{STORY_USER_ID}'; DELETE FROM fired_events WHERE user_id='{STORY_USER_ID}'",
    }
    sql = ops.get(STORY_ACTION, "")
    if sql:
        story_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
async function run() {{
  const sqls = `{sql}`.split(';').map(s=>s.trim()).filter(Boolean);
  for (const q of sqls) {{
    const r = await db.execute(q);
    if (r.rows?.length) {{
      console.log('columns:', r.columns.join(', '));
      r.rows.forEach((row,i) => {{
        const o={{}};r.columns.forEach((c,j)=>o[c]=row[j]);
        console.log(i+1, JSON.stringify(o));
      }});
    }} else {{
      console.log('rowsAffected:', r.rowsAffected);
    }}
  }}
}}
run().catch(e=>{{console.error(e.message);process.exit(1);}});
"""
        ss = GIT_DIR / "_story.mjs"
        ss.write_text(story_code)
        run(f"node {ss} 2>&1")
        ss.unlink(missing_ok=True)
        print(f"\nstory {STORY_ACTION} complete")
else:
    print("Story direct op skipped (RUN_STORY_DIRECT=False or STORY_USER_ID not set)")

# --- D. Division transfer review ---
TRANSFER_ID     = ""
TRANSFER_ACTION = "approved"   # "approved" or "rejected"
REJECT_REASON   = ""

if RUN_TRANSFER_REVIEW and TRANSFER_ID:
    body = {"id": TRANSFER_ID, "status": TRANSFER_ACTION}
    if REJECT_REASON:
        body["rejectReason"] = REJECT_REASON
    ok, _ = api_check(
        f"Transfer {TRANSFER_ACTION}: {TRANSFER_ID}",
        f"{base}/api/admin/division-transfer",
        method="PATCH", body=body,
        expected_status=200,
        cookie=session_cookie if "session_cookie" in dir() else "",
        check_key="ok", check_value=True,
    )
    if ok:
        print(f"\nTransfer {TRANSFER_ID} -> {TRANSFER_ACTION}")
else:
    print("Transfer review skipped (RUN_TRANSFER_REVIEW=False or TRANSFER_ID not set)")


---
## Util — Seed Achievement Master Data

In [ ]:
# ============================================================
# 実績マスターデータを DB に投入（migrate 後に一度だけ実行）
# ============================================================
RUN_SEED_ACHIEVEMENTS = True  # 初回のみ True にする

ACHIEVEMENT_MASTER_JS = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
const MASTER = [
  {{ key:'first_login',       title:'初期認証完了',    desc:'海蝕機関へようこそ。初回ログインを完了した。',                  icon:'◈', xp:50,   s:0 }},
  {{ key:'streak_3days',      title:'3日連続ログイン',  desc:'3日連続でシステムにアクセスした。',                           icon:'◉', xp:75,   s:0 }},
  {{ key:'streak_7days',      title:'週間任務継続',     desc:'7日間連続で接続した。献身的な機関員。',                        icon:'◉', xp:200,  s:0 }},
  {{ key:'streak_30days',     title:'鉄の意志',         desc:'30日間連続ログイン。機関への揺るぎない忠誠。',                 icon:'◆', xp:1000, s:1 }},
  {{ key:'level2_reached',    title:'クリアランス LV2', desc:'クリアランスレベル2に到達した。',                             icon:'▣', xp:0,    s:0 }},
  {{ key:'level3_reached',    title:'クリアランス LV3', desc:'クリアランスレベル3に到達した。',                             icon:'▣', xp:0,    s:0 }},
  {{ key:'level5_reached',    title:'最高機密取扱許可', desc:'クリアランスレベル5に到達した。蒼海計画が解放された。',       icon:'◆', xp:500,  s:1 }},
  {{ key:'xp_1000',           title:'1000XP到達',       desc:'累計1000XPを獲得した。',                                    icon:'⬡', xp:50,   s:0 }},
  {{ key:'xp_5000',           title:'熟練機関員',       desc:'累計5000XPを獲得した。',                                    icon:'⬡', xp:200,  s:0 }},
  {{ key:'first_mission',     title:'初任務完了',       desc:'初めてのミッションを完了した。',                             icon:'◈', xp:100,  s:0 }},
  {{ key:'mission_5',         title:'ベテラン収束員',   desc:'5件のミッションを完了した。',                                icon:'◉', xp:250,  s:0 }},
  {{ key:'chat_50',           title:'情報伝達者',       desc:'50通のメッセージを送信した。',                               icon:'◎', xp:75,   s:0 }},
  {{ key:'chat_500',          title:'機関の声',         desc:'500通のメッセージを送信した。',                              icon:'◎', xp:300,  s:1 }},
  {{ key:'division_transfer', title:'部門異動',         desc:'部門移動を申請・承認された。',                              icon:'◇', xp:100,  s:0 }},
  {{ key:'entity_researcher', title:'実体研究者',       desc:'10件以上の実体情報を閲覧した。',                             icon:'◆', xp:150,  s:0 }},
  {{ key:'high_anomaly',      title:'観測負荷：警戒',   desc:'異常スコアが50を超えた。観測される側になりつつある。',      icon:'〜', xp:0,    s:1 }},
];
async function seed() {{
  let count = 0;
  for (const a of MASTER) {{
    await db.execute({{ sql: 'INSERT OR IGNORE INTO achievements (id,key,title,description,icon,xp_reward,is_secret) VALUES (?,?,?,?,?,?,?)', args: ['ach-'+a.key, a.key, a.title, a.desc, a.icon, a.xp, a.s] }});
    count++;
  }}
  const r = await db.execute('SELECT COUNT(*) AS cnt FROM achievements');
  console.log('Seeded', count, 'definitions — total in DB:', r.rows[0][0]);
}}
seed().catch(e => {{ console.error(e.message); process.exit(1); }});
"""

if RUN_SEED_ACHIEVEMENTS:
    ach_path = GIT_DIR / "_seed_ach.mjs"
    ach_path.write_text(ACHIEVEMENT_MASTER_JS)
    result = run(f"node {ach_path} 2>&1")
    ach_path.unlink(missing_ok=True)
    if result.returncode == 0:
        print("\nAchievement master data seeded")
    else:
        print("\nFailed to seed achievements:", result.stdout)
else:
    print("RUN_SEED_ACHIEVEMENTS=False — skipped")


---
## Util — Mission Participant Management

In [ ]:
# ============================================================
# ミッション参加申請管理（STEP 8 実行後に使用）
# ============================================================
SHOW_PENDING_MISSIONS  = True  # pending 一覧を表示
APPROVE_PARTICIPANT_ID = ""    # 承認する参加申請ID（空=スキップ）
REJECT_PARTICIPANT_ID  = ""    # 却下する参加申請ID（空=スキップ）
REJECT_NOTE            = ""    # 却下理由

cookie = session_cookie if "session_cookie" in dir() else ""

if SHOW_PENDING_MISSIONS:
    ok, _ = api_check(
        "GET /api/admin/mission-participants (pending)",
        f"{base}/api/admin/mission-participants?status=pending",
        expected_status=200,
        cookie=cookie,
    )

if APPROVE_PARTICIPANT_ID:
    ok2, _ = api_check(
        f"APPROVE: {APPROVE_PARTICIPANT_ID}",
        f"{base}/api/admin/mission-participants",
        method="PATCH",
        body={"participantId": APPROVE_PARTICIPANT_ID, "status": "approved"},
        expected_status=200, cookie=cookie, check_key="ok", check_value=True,
    )

if REJECT_PARTICIPANT_ID:
    body = {"participantId": REJECT_PARTICIPANT_ID, "status": "rejected"}
    if REJECT_NOTE:
        body["note"] = REJECT_NOTE
    ok3, _ = api_check(
        f"REJECT: {REJECT_PARTICIPANT_ID}",
        f"{base}/api/admin/mission-participants",
        method="PATCH", body=body,
        expected_status=200, cookie=cookie, check_key="ok", check_value=True,
    )


---
## Util — 初期パズルデータシード

ARGの謎解きコンテンツ（暗号解読）のサンプルデータを投入する。
管理者画面（/admin/puzzles）でも追加可能。


In [ ]:

# ============================================================
# 初期パズルデータを投入（初回のみ）
# ============================================================
RUN_SEED_PUZZLES = True  # 初回のみ True

PUZZLE_SEED_JS = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});

const PUZZLES = [
  {{
    id: 'puz-001', slug: 'cipher-001',
    title: '第一暗号 — カエサル暗号',
    cipher_text: 'NDLVPF DJDQFH — NDLVKRNX\\n\\n海蝕現象の源について問え。\\n答えは機関の名に刻まれている。',
    answer: 'KAISHOKU',
    hint: '機関の正式名称をローマ字で',
    xp_reward: 75,
    clearance_req: 2,
  }},
  {{
    id: 'puz-002', slug: 'cipher-002',
    title: '第二暗号 — 逆読み',
    cipher_text: '以下の文字列を逆から読め:\\n\\nΩTEKERSBOA',
    answer: 'AOBSERKET',
    hint: 'そのままではなく逆向きに',
    xp_reward: 100,
    clearance_req: 2,
  }},
  {{
    id: 'puz-003', slug: 'cipher-003',
    title: '第三暗号 — 数字の意味',
    cipher_text: '観測点α-7の記録より:\\n\\n14-22-5-9-12\\n\\n各数字はアルファベットの何番目か。',
    answer: 'NVEIL',
    hint: 'A=1, B=2, ... Z=26',
    xp_reward: 150,
    clearance_req: 3,
  }},
];

async function seed() {{
  let count = 0;
  for (const p of PUZZLES) {{
    await db.execute({{
      sql: `INSERT OR IGNORE INTO puzzle_entries
              (id, slug, title, cipher_text, answer, hint, xp_reward, clearance_req, created_by)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'K-000-ADMIN')`,
      args: [p.id, p.slug, p.title, p.cipher_text, p.answer, p.hint, p.xp_reward, p.clearance_req],
    }});
    count++;
  }}
  const r = await db.execute('SELECT COUNT(*) AS cnt FROM puzzle_entries');
  console.log('Seeded', count, 'puzzles — total in DB:', r.rows[0][0]);
}}
seed().catch(e => {{ console.error(e.message); process.exit(1); }});
"""

if RUN_SEED_PUZZLES:
    ps = GIT_DIR / "_seed_puzzles.mjs"
    ps.write_text(PUZZLE_SEED_JS)
    result = run(f"node {ps} 2>&1")
    ps.unlink(missing_ok=True)
    if result.returncode == 0:
        print("\nPuzzle seed complete")
    else:
        print("\nPuzzle seed FAILED:", result.stdout)
else:
    print("RUN_SEED_PUZZLES=False — skipped")


---
## Util — ARGイベントスケジュール登録サンプル

管理者画面（/admin/event-schedule）でも同操作が可能。


In [ ]:

# ============================================================
# ARGイベントのサンプル登録
# ============================================================
RUN_SEED_EVENT = False  # True にして実行

if RUN_SEED_EVENT:
    import uuid
    from datetime import datetime, timezone, timedelta

    # 1時間後に発火するサンプルイベント
    trigger_time = (datetime.now(timezone.utc) + timedelta(hours=1)).isoformat()

    body = {
        "title": "【緊急】観測アラート Level-Ω",
        "description": "テスト用イベント",
        "triggerAt": trigger_time,
        "actions": [
            {
                "type": "notify",
                "notify_target": "all",
                "notify_title": "【緊急通達】海蝕指数が臨界値に接近",
                "notify_body": "全部門は直ちに警戒態勢に移行せよ。詳細はイベントページで確認。",
                "is_public": True,
                "public_title": "【緊急】次元臨界警報",
                "public_desc": "海蝕指数が観測史上最高値に達しつつあります。機関員は待機してください。",
            }
        ],
    }

    ok, _ = api_check(
        "POST /api/admin/event-schedule",
        f"{base}/api/admin/event-schedule",
        method="POST", body=body,
        expected_status=201,
        cookie=session_cookie if "session_cookie" in dir() else "",
        check_key="ok", check_value=True,
    )
    if ok:
        print(f"\nイベント登録完了: {trigger_time} に発火予定")
else:
    print("RUN_SEED_EVENT=False — skipped")


---
## Util — NPC DM チャンネル確認

登録済みユーザーの NPC DM チャンネル一覧を確認する。


In [ ]:

# ============================================================
# NPC DMチャンネルの状況を確認
# ============================================================
RUN_NPC_DM_CHECK = False

if RUN_NPC_DM_CHECK:
    check_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
async function check() {{
  const channels = await db.execute(`
    SELECT u.agent_id, d.npc_name, d.created_at,
           COUNT(cm.id) as msg_count
    FROM npc_dm_channels d
    JOIN users u ON u.id = d.user_id
    LEFT JOIN chat_messages cm ON cm.chat_id = ('npc-dm-' || lower(d.npc_name))
      AND cm.sender_id = d.user_id
    GROUP BY d.id
    ORDER BY d.created_at DESC
    LIMIT 20
  `);
  console.log('NPC DM Channels (' + channels.rows.length + '):');
  const SEP = '-'.repeat(55);
  console.log(SEP);
  channels.rows.forEach(r => {{
    const obj = {{}};
    channels.columns.forEach((c,i) => obj[c] = r[i]);
    console.log(\`  \${{String(obj.agent_id).padEnd(18)}} → \${{String(obj.npc_name).padEnd(8)}} | msgs=\${{obj.msg_count}} | \${{String(obj.created_at).slice(5,16)}}\`);
  }});
}}
check().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    cs = GIT_DIR / "_npc_dm_check.mjs"
    cs.write_text(check_code)
    run(f"node {cs} 2>&1")
    cs.unlink(missing_ok=True)
else:
    print("RUN_NPC_DM_CHECK=False — skipped")


---
## Util — 暗号解読統計

パズルごとの解答状況を表示する。


In [ ]:

# ============================================================
# 暗号解読統計を表示
# ============================================================
RUN_CIPHER_STATS = False

if RUN_CIPHER_STATS:
    stats_code = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});
const BAR = (n, max, w=20) => '█'.repeat(Math.round(n/Math.max(max,1)*w)).padEnd(w);
async function stats() {{
  const puzzles = await db.execute(`
    SELECT p.slug, p.title, p.xp_reward, p.clearance_req, p.is_active,
           COUNT(s.id) as solve_count
    FROM puzzle_entries p
    LEFT JOIN puzzle_solves s ON s.puzzle_id = p.id
    GROUP BY p.id
    ORDER BY solve_count DESC
  `);
  console.log('\\nCipher Solve Statistics:');
  console.log('-'.repeat(60));
  const maxSolves = Math.max(...puzzles.rows.map(r => Number(r[5])), 1);
  puzzles.rows.forEach(r => {{
    const [slug, title, xp, lv, active, cnt] = r;
    const bar = BAR(cnt, maxSolves);
    const status = active ? 'ACTIVE' : 'OFF   ';
    console.log(\`  [\${{status}}] \${{String(slug).padEnd(14)}} LV\${{lv}} +\${{xp}}XP  \${{bar}} \${{cnt}}\`);
    console.log(\`          \${{title}\`);
  }});
  console.log('-'.repeat(60));
  const total = await db.execute('SELECT COUNT(*) FROM puzzle_solves');
  console.log('Total solves:', total.rows[0][0]);
}}
stats().catch(e => {{ console.error(e.message); process.exit(1); }});
"""
    ss = GIT_DIR / "_cipher_stats.mjs"
    ss.write_text(stats_code)
    run(f"node {ss} 2>&1")
    ss.unlink(missing_ok=True)
else:
    print("RUN_CIPHER_STATS=False — skipped")


--- ## Util — World Data Seed（タグ拡張テーブル初期データ）

観測地点 / 裂孔 / 事案記録 / 作戦記録 / 封印プロトコル / 研究仮説のサンプルデータを投入する。  
管理者画面（`/admin/world-data`）からも入力可能。初回デプロイ後に一度だけ実行すること。

In [ ]:
# ============================================================
# タグ拡張テーブル（v5）の初期サンプルデータを投入
# ============================================================
# STEP 0 で RUN_SEED_WORLD_DATA = True にしてから実行
# ============================================================

if not RUN_SEED_WORLD_DATA:
    print("RUN_SEED_WORLD_DATA=False — skipped (enable in STEP 0)")
else:
    WORLD_SEED_JS = f"""
import {{ createClient }} from '@libsql/client';
const db = createClient({{ url: '{TURSO_URL}', authToken: '{TURSO_TOKEN}' }});

async function seed() {{
  // ── 観測地点（LOC- / RIFT-） ──────────────────────────
  const points = [
    {{
      id: 'LOC-OIT-001', type: 'location', name: '観測基地アルファ',
      name_short: 'α基地', lon: 131.5878, lat: 33.2517,
      city_code: '44201', city_name: '大分市',
      status: 'active', clearance_req: 0,
      gsi_current: null,
      description: '機関の主要観測拠点。大分市内に位置する。',
    }},
    {{
      id: 'RIFT-α7', type: 'rift_point', name: '観測点α-7',
      name_short: 'α-7', lon: 131.742, lat: 32.925,
      city_code: '44202', city_name: '別府市',
      status: 'monitoring', clearance_req: 1,
      gsi_current: 3.1,
      description: 'α系列の第7観測点。GSI値が安定して上昇中。',
    }},
    {{
      id: 'RIFT-β12', type: 'rift_point', name: '観測点β-12',
      name_short: 'β-12', lon: 131.800, lat: 32.842,
      city_code: '44202', city_name: '別府市',
      status: 'critical', clearance_req: 2,
      gsi_current: 4.8,
      description: 'K-17が最後の通信を送信した地点周辺。高度警戒中。',
    }},
  ];
  for (const p of points) {{
    await db.execute({{
      sql: `INSERT OR IGNORE INTO observation_points
            (id,type,name,name_short,lon,lat,city_code,city_name,status,clearance_req,gsi_current,description)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?)`,
      args: [p.id,p.type,p.name,p.name_short,p.lon,p.lat,p.city_code,p.city_name,p.status,p.clearance_req,p.gsi_current,p.description],
    }});
  }}
  console.log('observation_points: ' + points.length + ' records inserted');

  // ── 観測ログ（GSI- / SIG-） ────────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO observation_logs
          (id,type,title,observed_at,location_ref,gsi_value,gsi_baseline,severity,description)
          VALUES (?,?,?,?,?,?,?,?,?)`,
    args: ['GSI-RECORD-042','gsi','α-7 GSI急上昇記録','2026-03-05 02:17','RIFT-α7',4.8,1.2,'elevated',
           '観測点α-7にて3秒間のGSI急上昇を記録。直後にK-17の通信が途絶。'],
  }});
  await db.execute({{
    sql: `INSERT OR IGNORE INTO observation_logs
          (id,type,title,observed_at,location_ref,freq_band,amplitude_db,duration_sec,pattern_match,severity,description)
          VALUES (?,?,?,?,?,?,?,?,?,?,?)`,
    args: ['SIG-DELTA','signal','SIG-DELTA 傍受記録','2026-03-05 02:17','RIFT-α7',
           '12–18kHz',22,3,'ENT-001 PROXIMITY SIGNATURE','elevated',
           '人体発声なし。ENT-001の近接シグネチャと一致する広帯域EMノイズ。'],
  }});
  console.log('observation_logs: 2 records inserted');

  // ── 次元裂孔（CRK-） ──────────────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO dimension_cracks
          (id,name,lon,lat,location,status,severity,gsi_peak,first_detected,entity_emerged,clearance_req,description)
          VALUES (?,?,?,?,?,?,?,?,?,?,?,?)`,
    args: ['CRK-001','第一β裂孔',131.742,32.925,'別府湾沖','active','critical',8.2,
           '2026-02-14','["ENT-001","ENT-004"]',2,
           'β-12観測点付近に発生した最初の大型裂孔。複数の実体出現を確認。'],
  }});
  console.log('dimension_cracks: 1 record inserted');

  // ── 事案記録（CASE-IR-） ───────────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO case_reports
          (id,title,case_date,status,division_ref,entity_ref,location_ref,casualties,clearance_req,summary)
          VALUES (?,?,?,?,?,?,?,?,?,?)`,
    args: ['CASE-IR-031','収束研究所第一棟・第二棟閉鎖事案','2024-11-18','closed',
           'DIV-02','ENT-001','FAC-002',3,2,
           'FAC-002の前身施設でENT-001の群体化が発生。封印部門の介入により収束したが施設は閉鎖。'],
  }});
  console.log('case_reports: 1 record inserted');

  // ── 作戦記録（OP-） ────────────────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO operation_records
          (id,codename,title,op_date,end_date,status,division_json,target_ref,outcome,clearance_req,description,casualties)
          VALUES (?,?,?,?,?,?,?,?,?,?,?,?)`,
    args: ['OP-NIGHTFALL','NIGHTFALL','NIGHTFALL作戦 — β-12裂孔封印','2026-03-10','2026-03-13',
           'active','["DIV-02","DIV-05"]','CRK-001','partial',2,
           'β-12裂孔の封印を試みた収束部門・封印部門合同作戦。作戦継続中。',0],
  }});
  console.log('operation_records: 1 record inserted');

  // ── 封印プロトコル（PROTO-） ───────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO containment_protocols
          (id,codename,title,division_ref,status,threat_class,clearance_req,summary,steps_json)
          VALUES (?,?,?,?,?,?,?,?,?)`,
    args: ['PROTO-OMEGA','OMEGA','ΩプロトコルII — 大型裂孔緊急封印手順','DIV-05',
           'active','大型次元裂孔 / GSI 6.0σ超',3,
           '6.0σを超えるGSI値を持つ裂孔への緊急対応手順。封印部門最高機密。',
           JSON.stringify([
             {{step:1,title:'初動隔離',desc:'半径500m立入禁止。観測部門を展開。'}},
             {{step:2,title:'収束場展開',desc:'MOD-001-α を最大出力で起動。'}},
             {{step:3,title:'封印核投下',desc:'████████████（LV3以上）'}},
           ])],
  }});
  console.log('containment_protocols: 1 record inserted');

  // ── 研究仮説（THEORY-） ────────────────────────────────
  await db.execute({{
    sql: `INSERT OR IGNORE INTO research_theories
          (id,title,author_ref,division_ref,proposed_at,status,confidence,clearance_req,abstract,evidence_json)
          VALUES (?,?,?,?,?,?,?,?,?,?)`,
    args: ['THEORY-009','海蝕源単一起源仮説','AGT-N01','DIV-03','2026-01-15',
           'under_review',62,2,
           '全ての海蝕現象は単一の次元的起源から発生しているという仮説。SIGMAの観測データと一致する箇所が複数ある。',
           JSON.stringify([
             {{type:'observation',ref:'GSI-RECORD-042',desc:'複数観測点でのGSI値が特定周期で連動。'}},
             {{type:'entity',ref:'ENT-002',desc:'SIGMAが示唆する「源」の存在。'}},
           ])],
  }});
  console.log('research_theories: 1 record inserted');

  console.log('\nWorld data seed complete.');
}}

seed().catch(e => {{ console.error('Seed failed:', e.message); process.exit(1); }});
""";

    seed_path = GIT_DIR / "_world_seed.mjs"
    seed_path.write_text(WORLD_SEED_JS)
    print("Running world data seed...")
    result = run(f"node {seed_path} 2>&1")
    seed_path.unlink(missing_ok=True)
    if result.returncode != 0:
        raise RuntimeError(f"World seed failed:\n{result.stdout}\n{result.stderr}")
    print("\nWorld data seed complete.")
